In [1]:
import os
import sys
import io
import logging
import requests
import zipfile
import xml.etree.ElementTree as ET
from typing import Optional, Dict, List
from pathlib import Path
import pandas as pd
import datetime as dt
import pymysql
import FinanceDataReader as fdr

# ---------------------------------------------------------
# 기본 로깅 설정
# ---------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)


# ---------------------------------------------------------
# 0) 프로젝트 루트 자동 탐색 (DATA 폴더 기준)
# ---------------------------------------------------------
def add_repo_path():
    """프로젝트 루트를 자동 탐색하여 sys.path에 추가"""
    if '__file__' in globals():
        current = Path(__file__).resolve().parent
    else:
        current = Path.cwd()

    for parent in [current] + list(current.parents):
        if (parent / "DATA").exists():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            logger.info(f"Project root added: {parent}")
            return str(parent)

    fallback = r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast"
    if os.path.isdir(fallback):
        if fallback not in sys.path:
            sys.path.insert(0, fallback)
        logger.warning(f"Using fallback path: {fallback}")
        return fallback

    raise FileNotFoundError("DATA 폴더를 찾을 수 없습니다.")


try:
    project_root = add_repo_path()
    from DATA.stock_invest_function import get_db_host
except ImportError:
    logger.warning("stock_invest_function import 실패 - DB 정보를 직접 설정해야 합니다")


# ---------------------------------------------------------
# 1) corp_code 목록 불러오기 (DART corpCode.xml)
# ---------------------------------------------------------
def load_corp_code(api_key: str) -> pd.DataFrame:
    """
    DART에서 corpCode.zip을 내려받아
    corp_code, corp_name, stock_code 정보를 DataFrame으로 반환.
    """
    url = "https://opendart.fss.or.kr/api/corpCode.xml"
    params = {"crtfc_key": api_key}
    r = requests.get(url, params=params)
    r.raise_for_status()

    content_type = (r.headers.get("Content-Type") or "").lower()
    head_bytes = r.content[:4]  # ZIP 여부 판별용 (b'PK\\x03\\x04')

    # 1) 에러(XML) 응답인지 먼저 체크
    if ("xml" in content_type or "text" in content_type) and not head_bytes.startswith(b"PK"):
        try:
            root = ET.fromstring(r.text)
            status = root.findtext("status")
            message = root.findtext("message")
            if status != "000":
                raise RuntimeError(
                    f"[DART corpCode 오류] status={status}, message={message}"
                )
        except ET.ParseError:
            raise RuntimeError(
                f"[DART corpCode 오류] XML 파싱 실패. "
                f"Content-Type={content_type}, text={r.text[:200]}"
            )

    # 2) 정상: ZIP 파일 처리
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        xml_name = None
        for name in z.namelist():
            if name.lower().endswith(".xml"):
                xml_name = name
                break

        if xml_name is None:
            raise RuntimeError(
                f"[DART corpCode 오류] ZIP 안에 XML 파일이 없습니다. files={z.namelist()}"
            )

        with z.open(xml_name) as xml_file:
            tree = ET.parse(xml_file)
            root = tree.getroot()

    # 3) XML → DataFrame 변환
    rows = []
    for child in root.findall("list"):
        corp_code = child.findtext("corp_code")
        corp_name = child.findtext("corp_name")
        stock_code = child.findtext("stock_code")
        rows.append(
            {
                "corp_code": corp_code,
                "corp_name": corp_name,
                "stock_code": stock_code,
            }
        )

    df = pd.DataFrame(rows)
    df = df[df["stock_code"].notnull() & (df["stock_code"] != "")]
    df.reset_index(drop=True, inplace=True)
    return df


# ---------------------------------------------------------
# 2) FinanceDataReader 종목 코드로 corp_code 찾기
# ---------------------------------------------------------
def get_corp_info(corp_df: pd.DataFrame, stock_code: str) -> Optional[Dict]:
    """
    FinanceDataReader 형식의 종목코드(예: '005930')로
    corp_df에서 해당 기업의 corp_code, corp_name, stock_code 를 찾아 dict로 반환.
    """
    row = corp_df.loc[corp_df["stock_code"] == stock_code]
    if row.empty:
        return None

    row = row.iloc[0]
    return {
        "corp_code": row["corp_code"],
        "corp_name": row["corp_name"],
        "stock_code": row["stock_code"],
    }

def test_db_connection(db_info: dict) -> bool:
    """
    MariaDB 연결 테스트 함수.
    연결 성공하면 True, 실패하면 False 반환.
    """
    try:
        conn = pymysql.connect(
            host=db_info["host"],
            port=db_info["port"],
            user=db_info["user"],
            password=db_info["password"],
            database=db_info["database"],
            charset="utf8mb4",
            connect_timeout=5
        )
        conn.close()
        logger.info("DB 연결 성공")
        return True
    except Exception as e:
        logger.error(f"DB 연결 실패: {e}")
        return False



# ---------------------------------------------------------
# 3) 분기별 재무제표 수신 (fnlttSinglAcntAll)
#    - 먼저 CFS 시도, 없으면 OFS로 fallback
# ---------------------------------------------------------
def get_dart_fs_quarterly(api_key: str,
                          corp_code: str,
                          start_year: int,
                          end_year: int) -> pd.DataFrame:
    """
    DART 'fnlttSinglAcntAll' API를 사용하여 분기별 재무제표 수집.
    먼저 CFS(연결) 시도 → 자료 없으면 OFS(개별)로 자동 fallback.
    """

    def fetch_one_year(api_key, corp_code, year, fs_div):
        """특정 연도·fs_div로 조회하는 내부 함수"""
        url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"
        reprt_map = {
            "11013": ("Q1", "-03-31"),
            "11012": ("H1", "-06-30"),
            "11014": ("Q3", "-09-30"),
            "11011": ("FY", "-12-31"),
        }

        rows: List[Dict] = []

        for reprt_code, (quarter_label, date_suffix) in reprt_map.items():
            params = {
                "crtfc_key": api_key,
                "corp_code": corp_code,
                "bsns_year": str(year),
                "reprt_code": reprt_code,
                "fs_div": fs_div,
            }

            r = requests.get(url, params=params)
            r.raise_for_status()
            data = r.json()

            if data.get("status") != "000":
                continue   # 자료 없음 → 다음 보고서

            for item in data.get("list", []):
                row = {
                    "corp_code": item.get("corp_code"),
                    "bsns_year": int(item.get("bsns_year")),
                    "reprt_code": item.get("reprt_code"),
                    "sj_div": item.get("sj_div"),
                    "sj_nm": item.get("sj_nm"),
                    "account_id": item.get("account_id"),
                    "account_nm": item.get("account_nm"),
                    "thstrm_nm": item.get("thstrm_nm"),
                    "thstrm_amount": item.get("thstrm_amount"),
                    "quarter": quarter_label,
                }
                # 날짜
                try:
                    row["report_date"] = dt.datetime.strptime(
                        f"{year}{date_suffix}", "%Y-%m-%d"
                    ).date()
                except Exception:
                    row["report_date"] = None

                rows.append(row)

        return rows

    # 1) CFS 먼저
    all_rows: List[Dict] = []
    for year in range(start_year, end_year + 1):
        rows = fetch_one_year(api_key, corp_code, year, fs_div="CFS")
        if rows:
            all_rows.extend(rows)

    # 2) CFS 없으면 OFS로 재시도
    if len(all_rows) == 0:
        print("[INFO] CFS(연결) 자료 없음 → OFS(개별)로 재시도합니다.")
        for year in range(start_year, end_year + 1):
            rows = fetch_one_year(api_key, corp_code, year, fs_div="OFS")
            if rows:
                all_rows.extend(rows)

    if not all_rows:
        print("[WARN] CFS/OFS 모두 자료 없음")
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)

    # 금액 숫자 변환
    df["thstrm_amount"] = pd.to_numeric(df["thstrm_amount"], errors="coerce")

    df = df.sort_values(["bsns_year", "reprt_code", "account_nm"]).reset_index(drop=True)
    return df

def run_dart_fs_for_top_n(api_key: str,
                          db_info: dict,
                          start_year: int = 2015,
                          end_year: int = 2025,
                          top_n: int = 50,
                          batch_size: int = 10,
                          use_fdr_filter: bool = True,
                          table_name: str = "korea_fs_data_from_DART"):
    """
    1) DART corp 목록 로드
    2) FDR 시가총액 기준 상위 top_n 종목 선택
    3) 각 종목에 대해 DART 분기 재무 데이터를 수집
    4) 회사 batch_size개 단위로 DB에 저장
    5) 에러 발생 종목은 error_list에 기록

    반환:
        error_list: [(stock_code, corp_name, 에러메시지), ...]
    """

    # DB 연결 테스트
    if not test_db_connection(db_info):
        logger.error("DB 연결 실패로 작업을 중단합니다")
        return []

    logger.info("DB 연결 테스트 완료")

    # 1) DART 기업 목록 로드
    print("=" * 70)
    logger.info("[STEP 1] DART 기업 목록 로드 중...")

    corp_df = load_corp_code(api_key)

    # DART 상장사만 필터링
    corp_df = corp_df[
        corp_df["stock_code"].notna() &
        (corp_df["stock_code"] != "") &
        (corp_df["stock_code"].str.strip() != "")
    ].copy()
    corp_df["stock_code"] = corp_df["stock_code"].astype(str).str.zfill(6)

    logger.info(f"DART 상장사 필터링 완료: {len(corp_df)}개")

    # 2) FDR 기반 현재 상장사 + 시가총액 상위 N개 필터링
    if use_fdr_filter:
        logger.info("[STEP 1-2] FinanceDataReader로 현재 상장 종목 + 시가총액 상위 종목 필터링...")

        try:
            fdr_df = fdr.StockListing("KRX")
            fdr_df["Code"] = fdr_df["Code"].astype(str).str.zfill(6)

            # ETF/ETN/REIT/SPAC 제거
            exclude_types = ["ETF", "ETN", "REIT", "SPAC"]

            if "Type" in fdr_df.columns:
                before = len(fdr_df)
                fdr_df = fdr_df[~fdr_df["Type"].isin(exclude_types)].copy()
                logger.info(f"Type 기반 필터링: {before}개 -> {len(fdr_df)}개")
            else:
                logger.warning("FDR 데이터에 'Type' 컬럼 없음 - Name 기반 필터 사용")
                pattern = r"ETF|ETN|리츠|리트|스팩|SPAC"
                before = len(fdr_df)
                fdr_df = fdr_df[~fdr_df["Name"].str.contains(pattern, case=False, na=False)].copy()
                logger.info(f"Name 기반 필터링: {before}개 -> {len(fdr_df)}개")

            # 시가총액 기준 상위 N개
            if "Marcap" not in fdr_df.columns:
                raise RuntimeError("FDR 데이터에 'Marcap' 컬럼이 없습니다. 버전을 확인하세요.")

            fdr_df = fdr_df.dropna(subset=["Marcap"]).copy()
            fdr_df = fdr_df.sort_values("Marcap", ascending=False)

            fdr_top = fdr_df.head(top_n).copy()
            top_codes = set(fdr_top["Code"].tolist())
            logger.info(f"FDR 시가총액 상위 {top_n}개 코드 추출 완료")

            # DART corp_df와 조인
            before = len(corp_df)
            corp_df = corp_df[corp_df["stock_code"].isin(top_codes)].copy()
            logger.info(f"DART 상장사 중 시가총액 상위 {top_n} 교집합: {before}개 -> {len(corp_df)}개")

        except Exception as e:
            logger.error(f"FDR 필터링 실패: {e}")
            logger.warning("FDR 필터를 건너뛰고 DART 목록만 사용 (시가총액 필터 없음)")

    total_companies = len(corp_df)
    logger.info(f"최종 대상 기업(루프 대상): {total_companies}개")

    print("\n[상장사 샘플]")
    print(corp_df[["corp_name", "stock_code"]].head(10))
    print()

    # -------------------------------------------------
    # 3) 메인 루프: 회사별로 DART 재무제표 수집 + 배치 저장
    # -------------------------------------------------
    error_list = []
    batch_list: List[pd.DataFrame] = []
    batch_codes: List[str] = []

    target_cols = [
        'corp_code', 'bsns_year', 'reprt_code', 'sj_div', 'sj_nm',
        'account_id', 'account_nm', 'thstrm_nm', 'thstrm_amount',
        'quarter', 'report_date'
    ]

    processed_count = 0

    for idx, row in corp_df.iterrows():
        stock_code = row["stock_code"]
        corp_code = row["corp_code"]
        corp_name = row["corp_name"]

        logger.info(f"[{processed_count + 1}/{total_companies}] {corp_name}({stock_code}) 처리 중...")

        try:
            fs_df = get_dart_fs_quarterly(
                api_key=api_key,
                corp_code=corp_code,
                start_year=start_year,
                end_year=end_year,
            )

            if fs_df.empty:
                logger.warning(f"{corp_name}({stock_code}) : 재무데이터 없음 (fs_df empty)")
                processed_count += 1
                continue

            # 필요한 컬럼만 추출
            missing_cols = [c for c in target_cols if c not in fs_df.columns]
            if missing_cols:
                logger.warning(f"{corp_name}({stock_code}) : 필요한 컬럼 누락 - {missing_cols}")
                processed_count += 1
                continue

            fs_df_refined = fs_df[target_cols].copy()
            fs_df_refined["ticker"] = stock_code  # 이 batch 함수에서는 ticker를 미리 넣어둡니다.

            batch_list.append(fs_df_refined)
            batch_codes.append(stock_code)
            processed_count += 1

            # 배치 크기에 도달하면 DB에 저장
            if len(batch_list) >= batch_size:
                logger.info(f"[BATCH SAVE] 회사 {len(batch_list)}개 묶어서 DB 저장 시도...")
                try:
                    save_fs_batch_to_db(batch_list, db_info=db_info, table_name=table_name)
                    logger.info(f"[BATCH SAVE] 저장 완료 (회사 {len(batch_list)}개)")
                except Exception as be:
                    logger.error(f"[BATCH SAVE ERROR] 배치 저장 중 오류 발생: {be}")
                    # 배치에 포함된 종목 모두를 에러 리스트에 추가
                    for sc in batch_codes:
                        error_list.append((sc, "BATCH_ERROR", str(be)))
                finally:
                    # 배치 초기화
                    batch_list = []
                    batch_codes = []

        except Exception as e:
            logger.error(f"{corp_name}({stock_code}) 처리 중 오류 발생: {e}")
            error_list.append((stock_code, corp_name, str(e)))
            processed_count += 1
            continue

    # 마지막으로 남은 배치 처리
    if batch_list:
        logger.info(f"[FINAL BATCH SAVE] 남은 회사 {len(batch_list)}개 DB 저장 시도...")
        try:
            save_fs_batch_to_db(batch_list, db_info=db_info, table_name=table_name)
            logger.info(f"[FINAL BATCH SAVE] 저장 완료 (회사 {len(batch_list)}개)")
        except Exception as be:
            logger.error(f"[FINAL BATCH SAVE ERROR] 배치 저장 중 오류 발생: {be}")
            for sc in batch_codes:
                error_list.append((sc, "BATCH_ERROR", str(be)))

    logger.info(f"작업 완료. 총 기업 수: {total_companies}, 에러 기업 수: {len(error_list)}")

    if error_list:
        print("\n[에러 발생 종목 목록]")
        for sc, name, msg in error_list:
            print(f" - {sc} / {name} / {msg[:100]}")

    return error_list

def run_dart_fs_for_top_range(api_key: str,
                              db_info: dict,
                              start_year: int,
                              end_year: int,
                              top_start: int,
                              top_end: int,
                              batch_size: int = 10,
                              use_fdr_filter: bool = True,
                              table_name: str = "korea_fs_data_from_DART"):

    # 1) DB 연결 테스트
    if not test_db_connection(db_info):
        logger.error("DB 연결 실패로 작업 중단")
        return []

    # 2) DART corp 목록 로드
    corp_df = load_corp_code(api_key)
    corp_df = corp_df[corp_df["stock_code"].notnull()].copy()
    corp_df["stock_code"] = corp_df["stock_code"].astype(str).str.zfill(6)

    # 3) FDR 시총 데이터 로드
    if use_fdr_filter:
        fdr_df = fdr.StockListing("KRX")
        fdr_df["Code"] = fdr_df["Code"].astype(str).str.zfill(6)
        exclude = ["ETF","ETN","REIT","SPAC"]

        if "Type" in fdr_df.columns:
            fdr_df = fdr_df[~fdr_df["Type"].isin(exclude)].copy()

        # 시총 기준 정렬
        fdr_df = fdr_df.dropna(subset=["Marcap"])
        fdr_df = fdr_df.sort_values("Marcap", ascending=False)

        # 4) 범위 선택 (예: 51~100)
        fdr_range = fdr_df.iloc[top_start-1 : top_end]   # 1-indexed → 0-index 변환
        target_codes = set(fdr_range["Code"].tolist())

        print(f"[INFO] 시총 {top_start} ~ {top_end}위 기업 수: {len(target_codes)}")
    else:
        target_codes = set(corp_df["stock_code"].tolist())

    # DART corp_code 조인
    corp_df = corp_df[corp_df["stock_code"].isin(target_codes)].copy()

    # 기존 batch 저장 루틴 재사용
    error_list = run_dart_fs_for_stock_list(
        api_key=api_key,
        db_info=db_info,
        stock_code_list=list(corp_df["stock_code"]),
        start_year=start_year,
        end_year=end_year,
        batch_size=batch_size,
        table_name=table_name,
    )

    return error_list

# ---------------------------------------------------------
# 4) DB 저장 함수
# ---------------------------------------------------------
def save_fs_batch_to_db(batch_list: List[pd.DataFrame],
                        db_info: dict,
                        table_name: str = "korea_fs_data_from_DART"):
    """
    여러 회사의 fs_df_refined(DataFrame)를 한 번에 DB에 저장하는 배치 함수.

    batch_list: 각 원소가 다음 컬럼을 가진 DataFrame
        ['corp_code', 'bsns_year', 'reprt_code', 'sj_div', 'sj_nm',
         'account_id', 'account_nm', 'thstrm_nm', 'thstrm_amount',
         'quarter', 'report_date', 'ticker']
    """

    if not batch_list:
        return

    # 하나로 합치기
    df = pd.concat(batch_list, ignore_index=True)

    # 타입 정리
    df["bsns_year"] = pd.to_numeric(df["bsns_year"], errors="coerce").astype("Int64")
    df["quarter"] = df["quarter"].astype(str)
    df["thstrm_amount"] = pd.to_numeric(df["thstrm_amount"], errors="coerce")
    df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce").dt.date
    df["reprt_code"] = df["reprt_code"].astype(str)

    # PK에 들어가는 account_id 비어있으면 제거
    before = len(df)
    df = df[df["account_id"].notnull() & (df["account_id"] != "")]
    after = len(df)
    if before != after:
        logger.warning(f"[BATCH] account_id 없음으로 제거된 행: {before - after} rows")

    # NaN/NaT/<NA> → None
    df = df.where(pd.notnull(df), None)
    df = df.replace({pd.NA: None})
    df = df.replace({float('nan'): None})
    df = df.astype(object).where(df.notnull(), None)

    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4",
        autocommit=False,
    )

    try:
        with conn.cursor() as cur:
            # 테이블이 없으면 생성
            create_sql = f"""
            CREATE TABLE IF NOT EXISTS {table_name} (
                corp_code      VARCHAR(20)   NOT NULL,
                bsns_year      INT           NOT NULL,
                reprt_code     VARCHAR(10)   NOT NULL,
                quarter        VARCHAR(10)   NOT NULL,
                account_id     VARCHAR(100)  NOT NULL,

                sj_div         VARCHAR(10),
                sj_nm          VARCHAR(100),
                account_nm     VARCHAR(255),
                thstrm_nm      VARCHAR(50),
                thstrm_amount  DOUBLE,
                report_date    DATE,
                ticker         VARCHAR(20)   NOT NULL,

                PRIMARY KEY (corp_code, bsns_year, reprt_code, quarter, account_id)
            ) CHARACTER SET utf8mb4;
            """
            cur.execute(create_sql)

            insert_sql = f"""
            INSERT INTO {table_name} (
                corp_code, bsns_year, reprt_code, sj_div, sj_nm,
                account_id, account_nm, thstrm_nm, thstrm_amount,
                quarter, report_date, ticker
            ) VALUES (
                %(corp_code)s, %(bsns_year)s, %(reprt_code)s, %(sj_div)s, %(sj_nm)s,
                %(account_id)s, %(account_nm)s, %(thstrm_nm)s, %(thstrm_amount)s,
                %(quarter)s, %(report_date)s, %(ticker)s
            )
            ON DUPLICATE KEY UPDATE
                sj_div        = VALUES(sj_div),
                sj_nm         = VALUES(sj_nm),
                account_nm    = VALUES(account_nm),
                thstrm_nm     = VALUES(thstrm_nm),
                thstrm_amount = VALUES(thstrm_amount),
                report_date   = VALUES(report_date),
                ticker        = VALUES(ticker);
            """

            records = df.to_dict(orient="records")
            cur.executemany(insert_sql, records)

        conn.commit()
        logger.info(f"[BATCH] {len(df)} rows saved into {table_name}")

    except Exception as e:
        conn.rollback()
        logger.error(f"[BATCH] DB 저장 중 오류 발생: {e}")
        raise
    finally:
        conn.close()

def run_dart_fs_for_stock_list(api_key: str,
                               db_info: dict,
                               stock_code_list: list,
                               start_year: int = 2015,
                               end_year: int = 2025,
                               batch_size: int = 10,
                               table_name: str = "korea_fs_data_from_DART"):
    """
    지정한 stock_code 리스트(예: ['005930','000660', ...])에 대해서만
    DART 분기 재무제표를 수집하고, batch_size개 회사 단위로 DB에 저장.

    - api_key: DART API 키
    - db_info: MariaDB 접속 정보 딕셔너리
    - stock_code_list: 종목코드 리스트 (길이 N)
    - start_year, end_year: 재무제표 수집 연도 범위
    - batch_size: 몇 개 회사 단위로 DB에 저장할지 (기본 10)
    - table_name: 저장할 테이블 이름

    반환:
        error_list: [(stock_code, corp_name_or_reason, error_message), ...]
    """

    # 0) DB 연결 테스트
    if not test_db_connection(db_info):
        logger.error("DB 연결 실패로 작업을 중단합니다")
        return []

    logger.info("DB 연결 테스트 완료")

    # 1) DART 기업 목록 로드
    logger.info("[STEP 1] DART 기업 목록 로드 중...")
    corp_df = load_corp_code(api_key)

    # 상장사만 필터링 + stock_code 6자리 정규화
    corp_df = corp_df[
        corp_df["stock_code"].notna() &
        (corp_df["stock_code"] != "") &
        (corp_df["stock_code"].str.strip() != "")
    ].copy()
    corp_df["stock_code"] = corp_df["stock_code"].astype(str).str.zfill(6)

    logger.info(f"DART 상장사 필터링 완료: {len(corp_df)}개")

    # 2) 입력받은 stock_code 리스트 정규화 (중복 제거 + 6자리 패딩)
    normalized_codes = sorted(set(str(code).zfill(6) for code in stock_code_list))
    logger.info(f"사용자 지정 종목 수: {len(stock_code_list)}개 -> 정규화 후 {len(normalized_codes)}개")

    # corp_df에서 빠른 lookup을 위해 dict 생성 (stock_code -> (corp_code, corp_name))
    corp_map = {}
    for _, r in corp_df[["corp_code", "corp_name", "stock_code"]].iterrows():
        corp_map[r["stock_code"]] = (r["corp_code"], r["corp_name"])

    # 3) 메인 루프: 회사별 재무제표 수집 + 배치 저장
    error_list = []
    batch_list: List[pd.DataFrame] = []
    batch_codes: List[str] = []

    target_cols = [
        'corp_code', 'bsns_year', 'reprt_code', 'sj_div', 'sj_nm',
        'account_id', 'account_nm', 'thstrm_nm', 'thstrm_amount',
        'quarter', 'report_date'
    ]

    total = len(normalized_codes)
    processed = 0

    for stock_code in normalized_codes:
        processed += 1

        if stock_code not in corp_map:
            msg = "DART corp_code를 찾을 수 없음"
            logger.warning(f"[{processed}/{total}] {stock_code}: {msg}")
            error_list.append((stock_code, "NOT_FOUND_IN_DART", msg))
            continue

        corp_code, corp_name = corp_map[stock_code]
        logger.info(f"[{processed}/{total}] {corp_name}({stock_code}) 처리 중...")

        try:
            # 3-1) 재무데이터 수신
            fs_df = get_dart_fs_quarterly(
                api_key=api_key,
                corp_code=corp_code,
                start_year=start_year,
                end_year=end_year,
            )

            if fs_df.empty:
                msg = "재무데이터 없음 (fs_df empty)"
                logger.warning(f"{corp_name}({stock_code}) : {msg}")
                error_list.append((stock_code, corp_name, msg))
                continue

            # 3-2) 필요한 컬럼 체크
            missing_cols = [c for c in target_cols if c not in fs_df.columns]
            if missing_cols:
                msg = f"필요한 컬럼 누락: {missing_cols}"
                logger.warning(f"{corp_name}({stock_code}) : {msg}")
                error_list.append((stock_code, corp_name, msg))
                continue

            # 3-3) 정제 후 배치 리스트에 추가
            fs_df_refined = fs_df[target_cols].copy()
            fs_df_refined["ticker"] = stock_code  # 여기서 ticker 추가

            batch_list.append(fs_df_refined)
            batch_codes.append(stock_code)

            # 3-4) 배치 크기에 도달하면 DB에 저장
            if len(batch_list) >= batch_size:
                logger.info(f"[BATCH SAVE] 회사 {len(batch_list)}개 묶어서 DB 저장 시도...")
                try:
                    save_fs_batch_to_db(batch_list, db_info=db_info, table_name=table_name)
                    logger.info(f"[BATCH SAVE] 저장 완료 (회사 {len(batch_list)}개)")
                except Exception as be:
                    logger.error(f"[BATCH SAVE ERROR] 배치 저장 중 오류 발생: {be}")
                    for sc in batch_codes:
                        error_list.append((sc, "BATCH_ERROR", str(be)))
                finally:
                    batch_list = []
                    batch_codes = []

        except Exception as e:
            logger.error(f"{corp_name}({stock_code}) 처리 중 오류 발생: {e}")
            error_list.append((stock_code, corp_name, str(e)))
            continue

    # 4) 마지막으로 남은 배치 처리
    if batch_list:
        logger.info(f"[FINAL BATCH SAVE] 남은 회사 {len(batch_list)}개 DB 저장 시도...")
        try:
            save_fs_batch_to_db(batch_list, db_info=db_info, table_name=table_name)
            logger.info(f"[FINAL BATCH SAVE] 저장 완료 (회사 {len(batch_list)}개)")
        except Exception as be:
            logger.error(f"[FINAL BATCH SAVE ERROR] 배치 저장 중 오류 발생: {be}")
            for sc in batch_codes:
                error_list.append((sc, "BATCH_ERROR", str(be)))

    logger.info(f"작업 완료. 지정 종목 수: {total}, 에러 종목 수: {len(error_list)}")

    if error_list:
        print("\n[에러 발생 종목 목록]")
        for sc, name, msg in error_list:
            print(f" - {sc} / {name} / {msg[:100]}")

    return error_list




2025-11-22 18:09:33 [INFO] Project root added: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
2025-11-22 18:09:49 [WARNING] From C:\Users\82108\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\losses.py:2976: The name tf.losses.sparse_softmax_cross_entropy is deprecated. Please use tf.compat.v1.losses.sparse_softmax_cross_entropy instead.



In [2]:
API_KEY = "50424484a46daa88b34fcf875f40ca12b79e1fc1"   # ← 본인 키로 교체하세요
stock_code = "000660"                      # 예: 삼성전자 (FinanceDataReader 코드 형식)

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user' : 'stox7412',
    'password' : 'Apt106503!~',
    'database': 'investar'
}

# error_list = run_dart_fs_for_top_n(
#     api_key=API_KEY,
#     db_info=db_info,
#     start_year=2015,
#     end_year=2025,
#     top_n=50,        # 시가총액 상위 50개
#     batch_size=10,   # 10개 회사씩 몰아서 저장
#     use_fdr_filter=True,
#     table_name="korea_fs_data_from_DART",
# )

In [3]:
error_list = run_dart_fs_for_top_range(
    api_key=API_KEY,
    db_info=db_info,
    start_year=2015,
    end_year=2025,
    top_start=400,
    top_end=1000,
    batch_size=10,
    use_fdr_filter=True,
    table_name="korea_fs_data_from_DART",
)

2025-11-22 18:09:50 [INFO] DB 연결 성공
2025-11-22 18:09:53 [INFO] DB 연결 성공
2025-11-22 18:09:53 [INFO] DB 연결 테스트 완료
2025-11-22 18:09:53 [INFO] [STEP 1] DART 기업 목록 로드 중...


[INFO] 시총 400 ~ 1000위 기업 수: 601


2025-11-22 18:09:56 [INFO] DART 상장사 필터링 완료: 3913개
2025-11-22 18:09:56 [INFO] 사용자 지정 종목 수: 586개 -> 정규화 후 586개
2025-11-22 18:09:56 [INFO] [1/586] 경방(000050) 처리 중...
2025-11-22 18:10:02 [INFO] [2/586] 하이트진로홀딩스(000140) 처리 중...
2025-11-22 18:10:07 [INFO] [3/586] 노루홀딩스(000320) 처리 중...
2025-11-22 18:10:12 [INFO] [4/586] 한화손해보험(000370) 처리 중...
2025-11-22 18:10:16 [INFO] [5/586] 롯데손해보험(000400) 처리 중...


[INFO] CFS(연결) 자료 없음 → OFS(개별)로 재시도합니다.


2025-11-22 18:10:21 [INFO] [6/586] 대원강업(000430) 처리 중...
2025-11-22 18:10:26 [INFO] [7/586] 시알홀딩스(000480) 처리 중...
2025-11-22 18:10:32 [INFO] [8/586] 대동(000490) 처리 중...
2025-11-22 18:10:37 [INFO] [9/586] 삼일제약(000520) 처리 중...
2025-11-22 18:10:42 [INFO] [10/586] 흥국화재(000540) 처리 중...


...

[WARN] CFS/OFS 모두 자료 없음
[INFO] CFS(연결) 자료 없음 → OFS(개별)로 재시도합니다.


2025-11-22 18:47:44 [WARNING] 휴온스(243070) : 재무데이터 없음 (fs_df empty)
2025-11-22 18:47:44 [INFO] [441/586] 신흥에스이씨(243840) 처리 중...


[WARN] CFS/OFS 모두 자료 없음


2025-11-22 18:47:45 [ERROR] 신흥에스이씨(243840) 처리 중 오류 발생: ('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None, 10054, None))
2025-11-22 18:47:45 [INFO] [442/586] 와이엠티(251370) 처리 중...
2025-11-22 18:47:45 [ERROR] 와이엠티(251370) 처리 중 오류 발생: ('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None, 10054, None))
2025-11-22 18:47:45 [INFO] [443/586] 펌텍코리아(251970) 처리 중...
2025-11-22 18:47:45 [ERROR] 펌텍코리아(251970) 처리 중 오류 발생: ('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None, 10054, None))
2025-11-22 18:47:45 [INFO] [444/586] 샘씨엔에스(252990) 처리 중...
2025-11-22 18:47:45 [ERROR] 샘씨엔에스(252990) 처리 중 오류 발생: ('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None, 10054, None))
2025-11-22 18:47:45 [INFO] [445/586] 네오셈(253590) 처리 중...
2025-11-22 18:47:45 [ERROR] 네오셈(253590) 처리 중 오류 발생: ('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None


[에러 발생 종목 목록]
 - 0015G0 / 그린광학 / 재무데이터 없음 (fs_df empty)
 - 002960 / 한국쉘석유 / HTTPSConnectionPool(host='opendart.fss.or.kr', port=443): Max retries exceeded with url: /api/fnlttS
 - 094800 / 맵스리얼티 / 재무데이터 없음 (fs_df empty)
 - 126700 / 하이비젼시스템 / HTTPSConnectionPool(host='opendart.fss.or.kr', port=443): Max retries exceeded with url: /api/fnlttS
 - 140610 / 엔솔바이오사이언스 / 재무데이터 없음 (fs_df empty)
 - 230360 / 에코마케팅 / 재무데이터 없음 (fs_df empty)
 - 234030 / 싸이닉솔루션 / 재무데이터 없음 (fs_df empty)
 - 234080 / JW생명과학 / 재무데이터 없음 (fs_df empty)
 - 234340 / 헥토파이낸셜 / 재무데이터 없음 (fs_df empty)
 - 235980 / 메드팩토 / 재무데이터 없음 (fs_df empty)
 - 236200 / 슈프리마 / 재무데이터 없음 (fs_df empty)
 - 237880 / 클리오 / 재무데이터 없음 (fs_df empty)
 - 241590 / 화승엔터프라이즈 / 재무데이터 없음 (fs_df empty)
 - 241770 / 메카로 / 재무데이터 없음 (fs_df empty)
 - 243070 / 휴온스 / 재무데이터 없음 (fs_df empty)
 - 243840 / 신흥에스이씨 / ('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None, 10054, None
 - 251370 / 와이엠티 / ('Connection aborted.', ConnectionResetE

In [4]:
my_codes = ["051910", "035420", "005380", "006400", "035720",
            "000270", "207940", "068270", "042700", "043150",
            "131290", "006910", "140860", "095610", "001440",
            "000500", "004000", "010120", "068270", "058470"]  # 삼성전자, 하이닉스, NAVER, LG화학 등

error_list = run_dart_fs_for_stock_list(
    api_key=API_KEY,
    db_info=db_info,
    stock_code_list=my_codes,
    start_year=2015,
    end_year=2025,
    batch_size=10,   # 10개 모이면 저장 (여기서는 4개라 마지막에 한 번에 저장)
    table_name="korea_fs_data_from_DART",
)

2025-11-22 18:47:52 [INFO] DB 연결 성공
2025-11-22 18:47:52 [INFO] DB 연결 테스트 완료
2025-11-22 18:47:52 [INFO] [STEP 1] DART 기업 목록 로드 중...


ConnectionError: ('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None, 10054, None))

In [25]:

test_sample_path = r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\Results\Korea"

# 저장할 전체 파일 경로 만들기
output_path = os.path.join(test_sample_path, "isd_sample_data.xlsx")

# 필터링
test_df = fs_df[fs_df['sj_nm'] == '손익계산서']

# 저장
test_df.to_excel(output_path, index=False)

print(f"[INFO] 저장 완료: {output_path}")

[INFO] 저장 완료: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\Results\Korea\isd_sample_data.xlsx


In [27]:
test_df

,corp_code,bsns_year,reprt_code,sj_div,sj_nm,account_id,account_nm,account_detail,thstrm_nm,thstrm_amount,frmtrm_nm,frmtrm_amount,fs_div,fs_nm,quarter,report_date
101,00126380,2015,11011,IS,손익계산서,ifrs_ProfitLossFromContinuingOperations,계속영업이익(손실),-,제 47 기,1.906014e+13,제 46 기,2.339436e+13,None,None,FY,2015-12-31
102,00126380,2015,11011,IS,손익계산서,ifrs_FinanceCosts,금융비용,-,제 47 기,1.003177e+13,제 46 기,7.294002e+12,None,None,FY,2015-12-31
103,00126380,2015,11011,IS,손익계산서,ifrs_FinanceIncome,금융수익,-,제 47 기,1.051488e+13,제 46 기,8.259829e+12,None,None,FY,2015-12-31
104,00126380,2015,11011,IS,손익계산서,ifrs_BasicEarningsLossPerShare,기본주당이익(손실) (단위:원),-,제 47 기,1.263050e+05,제 46 기,1.531050e+05,None,None,FY,2015-12-31
105,00126380,2015,11011,IS,손익계산서,dart_OtherLosses,기타비용,-,제 47 기,3.723434e+12,제 46 기,2.259737e+12,None,None,FY,2015-12-31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6882,00126380,2024,11011,IS,손익계산서,dart_OperatingIncomeLoss,영업이익,-,제 56 기,3.272596e+13,제 55 기,6.566976e+12,None,None,FY,2024-12-31
6883,00126380,2024,11011,IS,손익계산서,ifrs-full_ProfitLossAttributableToOwnersOfParent,지배기업 소유지분,-,제 56 기,3.362136e+13,제 55 기,1.447340e+13,None,None,FY,2024-12-31
6884,00126380,2024,11011,IS,손익계산서,ifrs-full_ShareOfProfitLossOfAssociatesAndJoin...,지분법이익,-,제 56 기,7.510440e+11,제 55 기,8.875500e+11,None,None,FY,2024-12-31
6885,00126380,2024,11011,IS,손익계산서,dart_TotalSellingGeneralAdministrativeExpenses,판매비와관리비,-,제 56 기,8.158267e+13,제 55 기,7.197994e+13,None,None,FY,2024-12-31
